# Tutorial 9 · SHAP for global and local explanations

AI in Rehabilitation and Healthcare (376-1229-00L) · HS 2026 · Explainable machine learning II: SHAP

In this notebook you will:

1. Load the **activity recognition** feature dataset (`features_df_10s_3s.csv`).
2. Train a **black-box classifier** (Random Forest).
3. Use **SHAP (SHapley Additive exPlanations)** to:
   - Compute **global** feature importance.
   - Visualize **feature effects** (summary & dependence plots).
   - Inspect **local explanations** for individual windows.
   - Compare **correct vs incorrect** predictions.
   - See how **correlated features share** the credit, and compare with **permutation importance** on held-out people.

> **Goal:** Understand what the model has learned globally, and how it makes decisions for specific activity windows, using Shapley-value based explanations.


**Evaluate before you explain.** As in Tutorial 8, the model is scored on people it never saw, beside a baseline, before the first SHAP value.

**Data.** `features_df_10s_3s.csv` is computed from the public MotionSense dataset (iPhone accelerometer and gyroscope at 50 Hz, 24 people, 6 activities) with Tutorial 2's recipe: a 10 Hz low-pass filter, then ten features per axis on 10-second windows with a 3-second step. Cite it as: Malekzadeh M, Clegg RG, Cavallaro A, Haddadi H. Mobile sensor data anonymization. *IoTDI '19*, ACM, 2019, pp. 49–58. doi:10.1145/3302505.3310068.

**With the course coach.** Run *T9.00* with your code from SCAI Learning (*Your data → Connect a notebook*). The coach then sees the code of each cell you run, and the *Report* cells send it numbers about your split, your scores and your model, never your data and never a cell's output. Ask it about a cell by its name, for example *"why does T9.05 split by person?"*. Without a code the notebook runs the same and sends nothing.


In [ ]:
# T9.00 Connect the course coach (optional)
# In SCAI Learning, open Your data → Connect a notebook and copy the six-letter code. Paste it
# between the quotes and run this cell. Left empty, nothing is installed and nothing is sent.
# Connected, the coach receives the code of each cell you run and the numbers the Report cells
# compute: never your data, never a cell's output.
COACH_CODE = ""
coach = globals().get("coach")          # connected already with the app's snippet? keep it
if COACH_CODE and coach is None:
    %pip install -q https://app.scai-learning.ch/companion.tar.gz
    import rehab_coach as coach
    coach.connect(COACH_CODE)
    coach.watch()
print("Coach connected: the cells you run from here on reach it." if coach else "Working without the coach.")


## 1. Setup and load the dataset


In [ ]:
# T9.01 Install shap
%pip install -q shap


In [ ]:
# T9.02 Mount Google Drive
# Colab only. On Jupyter on your own computer, skip this cell and set data_path below to your folder.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# T9.03 Load the feature table
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import shap

RANDOM_STATE = 42

# --- Load the featured dataset ---
data_path = Path("/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/")   # Jupyter on your own computer: the folder that holds the file
load_path = data_path / "features_df_10s_3s.csv"

print(f"Loading data from: {load_path}")
features_df = pd.read_csv(load_path)
print("Shape:", features_df.shape)
display(features_df.head())


## 2. Prepare data


In [ ]:
# T9.04 Labels, features and people
# Encode labels (activity) into numeric values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(features_df["activity"])

# Extract features (drop metadata columns)
meta_cols = ["activity", "subject", "trial", "window"]
feature_cols = [c for c in features_df.columns if c not in meta_cols]

X = features_df[feature_cols].to_numpy().astype(float)
subjects = features_df["subject"].to_numpy()

print("Number of classes:", len(label_encoder.classes_))
print("Classes:", label_encoder.classes_)
print("Number of features:", len(feature_cols))


### 2.1 Subject-wise train–test split


In [ ]:
# T9.05 Split by person
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=subjects))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

subjects_train = subjects[train_idx]
subjects_test  = subjects[test_idx]

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Unique subjects in train:", np.unique(subjects_train))
print("Unique subjects in test :", np.unique(subjects_test))
print("People on both sides:", len(set(subjects_train) & set(subjects_test)))


In [ ]:
# T9.06 Report the split and the baseline
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T9.00).
if coach:
    coach.baseline(y_train)                     # the zero-rate score: always the commonest activity
    coach.report((train_idx, test_idx), subjects, y, tutorial="T9")   # opens the coach's explanation gate if the split is clean
else:
    print("Not connected: nothing sent. Run T9.00 with your code to use the coach.")


### 2.2 Standardize features (helpful for SHAP plots, though RF doesn’t require it)


In [ ]:
# T9.07 Standardise on the training people
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Scaled train shape:", X_train_s.shape)


## 3. Train the Random Forest model


In [ ]:
# T9.08 Train the black box and score it
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

model.fit(X_train_s, y_train)

proba_test = model.predict_proba(X_test_s)
pred_test  = model.predict(X_test_s)

acc = accuracy_score(y_test, pred_test)
f1  = f1_score(y_test, pred_test, average="macro")

from sklearn.metrics import balanced_accuracy_score
bal = balanced_accuracy_score(y_test, pred_test)
values, counts = np.unique(y_train, return_counts=True)
print(f"Random Forest — Accuracy: {acc:.3f}, balanced accuracy: {bal:.3f}, F1-macro: {f1:.3f}")
print(f"Zero-rate (always '{label_encoder.classes_[values[counts.argmax()]]}') accuracy: {(y_test == values[counts.argmax()]).mean():.3f}")
print("\nClassification report:")
print(classification_report(y_test, pred_test, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test, pred_test)
plt.figure(figsize=(6,4))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# T9.09 Report the model's scores
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T9.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, model=model, tutorial="T9",
                 scores={"accuracy": acc, "balanced_accuracy": bal, "f1_macro": f1})
else:
    print("Not connected: nothing sent. Run T9.00 with your code to use the coach.")


## 4. SHAP: SHapley Additive exPlanations

**Key idea:**

- Each prediction is written as a sum of **feature contributions** (Shapley values) relative to a baseline.
- For tree models (like Random Forest), `shap.TreeExplainer` computes SHAP values efficiently.
- We can look at:
  - **Global** patterns (which features are important overall, and how they affect predictions).
  - **Local** explanations (for one specific window).


In [ ]:
# T9.10 Set up SHAP
shap.initjs()

RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)


### 4.1 Background sample for SHAP (used as reference distribution)

The background stands for "a feature is absent": SHAP replaces a feature by values drawn from it. It comes from the **training** people; drawn from the test people, the explanation would lean on the data it is meant to be judged by.


In [ ]:
# T9.11 Background sample
bg_size = min(500, X_train_s.shape[0])
bg_idx = rng.choice(X_train_s.shape[0], size=bg_size, replace=False)
X_bg = X_train_s[bg_idx]

# Let SHAP pick the right explainer for our tree model
explainer = shap.Explainer(model, X_bg, feature_names=feature_cols)


### 4.2 Choose a subset of test samples to explain (for speed & readability)


In [ ]:
# T9.12 Compute SHAP values
test_size_for_shap = min(2000, X_test_s.shape[0])
test_idx_shap = rng.choice(X_test_s.shape[0], size=test_size_for_shap, replace=False)

X_test_shap = X_test_s[test_idx_shap]
y_test_shap = y_test[test_idx_shap]

# Compute SHAP values
shap_exp = explainer(X_test_shap)

vals = np.array(shap_exp.values)   # may be (n_samples, n_classes, n_features) or (n_samples, n_features)
print("shap_exp.values.shape:", vals.shape)
print("shap_exp.data.shape  :", np.array(shap_exp.data).shape)
print("n_features:", len(feature_cols))


### 4.3 Summary plot (handle multiclass)


In [ ]:
# T9.13 Summary plot for one activity
activity_names = list(label_encoder.classes_)
print("Activities:", activity_names)

# For multiclass, shap_exp has an "output" axis.
# We pick one activity class index to inspect, e.g. the first:
target_class_idx = 0
target_class_name = activity_names[target_class_idx]
print("Inspecting SHAP for class:", target_class_name)

# Take a view of the Explanation for that class.
# The `...` keeps the sample and feature axes; the last index picks the class/output.
exp_class = shap_exp[..., target_class_idx]

# Global dot summary
shap.plots.beeswarm(exp_class)


### 4.4 Bar summary (global importance only)


In [ ]:
# T9.14 Bar summary
shap.plots.bar(exp_class)


### 4.5 Dependence plot for the top feature

`exp_class.feature_names[0]` would be the **first column**, not the most important one. Rank the features by mean |SHAP| for this class and take the first of that ranking.


In [ ]:
# T9.15 Dependence plot for the top feature
mean_abs = np.abs(np.array(exp_class.values)).mean(axis=0)
top_feature = exp_class.feature_names[int(mean_abs.argmax())]
print("Top feature for", target_class_name, "by mean |SHAP|:", top_feature)

shap.plots.scatter(exp_class[:, top_feature])


## 5. Local explanation for one prediction
**Additivity.** The base value plus the SHAP values of one window add up to the model's output for that class: its predicted probability. That holds for wrong predictions too; it says what the model did, not whether it was right.


In [ ]:
# T9.16 Local explanation for one window
# Choose a random test point from the SHAP subset
i = rng.randint(0, exp_class.values.shape[0])

x0_true_idx = y_test_shap[i]
x0_pred_idx = model.predict(X_test_shap[i:i+1])[0]

true_label = label_encoder.inverse_transform([x0_true_idx])[0]
pred_label = label_encoder.inverse_transform([x0_pred_idx])[0]

print("Index in SHAP subset:", i)
print("True activity    :", true_label)
print("Predicted activity:", pred_label)
print("Explaining for class:", target_class_name)

# Local explanation for this sample and chosen class
exp_i = exp_class[i]

shap.plots.waterfall(exp_i)

base_value = float(np.ravel(exp_i.base_values)[0]) if np.ndim(exp_i.base_values) else float(exp_i.base_values)
total = base_value + float(np.sum(exp_i.values))
print(f"Base value {base_value:.3f} + SHAP values {np.sum(exp_i.values):+.3f} = {total:.3f}; "
      f"the forest's probability for '{target_class_name}': {model.predict_proba(X_test_shap[i:i+1])[0, target_class_idx]:.3f}")


## 6. Correct vs incorrect examples


In [ ]:
# T9.17 Correct against incorrect
# Predictions on the SHAP subset
pred_shap = model.predict(X_test_shap)

correct_idx = np.where(pred_shap == y_test_shap)[0]
wrong_idx   = np.where(pred_shap != y_test_shap)[0]

print("Correct in SHAP subset:", len(correct_idx), " Wrong:", len(wrong_idx))

def show_local_shap(idx, title=""):
    true_idx = y_test_shap[idx]
    pred_idx = pred_shap[idx]

    true_label = label_encoder.inverse_transform([true_idx])[0]
    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    print("="*60)
    print(title)
    print("Index:", idx)
    print("True activity    :", true_label)
    print("Predicted activity:", pred_label)

    # Explain for the predicted class of this instance
    pred_class_pos = activity_names.index(pred_label)
    exp_for_pred = shap_exp[..., pred_class_pos]  # Explanation for that class
    shap.plots.waterfall(exp_for_pred[idx])

if len(correct_idx) > 0:
    show_local_shap(correct_idx[0], "CORRECT example")

if len(wrong_idx) > 0:
    show_local_shap(wrong_idx[0], "WRONG example")


## 7. Correlated features share the credit

Many of these features measure the same movement: a standard deviation, a root mean square and a peak-to-peak range of one axis rise and fall together. When two features carry the same signal, SHAP splits the credit between them, and either can look less important than the signal is. Check how correlated the top features are before you rank them.


In [ ]:
# T9.18 How correlated are the top features?
vals = np.array(exp_class.values)            # (n_samples, n_features), for this class
feat_names = list(exp_class.feature_names)
order = np.argsort(-np.abs(vals).mean(axis=0))
top_feats = [feat_names[i] for i in order[:10]]

corr = pd.DataFrame(X_train_s, columns=feature_cols)[top_feats].corr()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().abs().sort_values(ascending=False)
max_corr = float(pairs.iloc[0])
print("Top 10 features by mean |SHAP| for", target_class_name, ":", top_feats)
print("Most correlated pairs among them (|r|, training people):")
display(pairs.head(5).round(2).to_frame("|r|"))


### 7.2 Visual interaction grids 

Now we create a small grid of dependence plots:
- rows = top 3 "main" features
- columns = 3 candidate “interaction” features


In [ ]:
# T9.19 Dependence plots coloured by a second feature
# Pick a small subset of features to visualize interactions
main_feats = top_feats[:2]
inter_feats = top_feats[2:4]  # two other top features; you can pick different ones

for main_f in main_feats:
    for inter_f in inter_feats:
        print(f"\nDependence plot: main={main_f}, colored by={inter_f}")
        shap.dependence_plot(
            main_f,
            vals,
            np.array(exp_class.data),
            feature_names=feat_names,
            interaction_index=inter_f  # color encodes potential interaction with inter_f
        )


### Optional: a simple numeric “interaction score” for pairs

We can also compute a rough interaction score for a pair (main, inter) by measuring how much the relationship between the main feature and its SHAP value changes when we bucket by the interacting feature.


In [ ]:
# T9.20 A rough interaction score
def rough_interaction_score(main_f, inter_f, exp_class, n_bins=4):
    """
    Heuristic: For each bin of inter_f, fit a simple linear relationship
    between main_f and its SHAP value and measure variation between bins.
    Bigger variation → stronger interaction.
    """
    vals = np.array(exp_class.values)
    X_data = np.array(exp_class.data)
    feat_names = list(exp_class.feature_names)

    i_main = feat_names.index(main_f)
    i_inter = feat_names.index(inter_f)

    x_main = X_data[:, i_main]
    x_inter = X_data[:, i_inter]
    shap_main = vals[:, i_main]

    # Bin inter_f
    q = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(x_inter, q)
    edges[0], edges[-1] = -np.inf, np.inf
    idx = np.digitize(x_inter, edges) - 1

    slopes = []
    for b in range(n_bins):
        m = idx == b
        if m.sum() < 5:
            continue
        xm = x_main[m]
        ym = shap_main[m]
        # simple linear regression slope (cov/var)
        v = np.var(xm)
        if v == 0:
            continue
        slope = np.cov(xm, ym)[0, 1] / v
        slopes.append(slope)

    if len(slopes) <= 1:
        return 0.0

    return float(np.std(slopes))

# Compute rough interaction scores for top feature pairs
pairs = []
for main_f in main_feats:
    for inter_f in inter_feats:
        if main_f == inter_f:
            continue
        score = rough_interaction_score(main_f, inter_f, exp_class)
        pairs.append((main_f, inter_f, score))

pairs_df = pd.DataFrame(pairs, columns=["main_feature", "inter_feature", "rough_interaction_score"])
pairs_df.sort_values("rough_interaction_score", ascending=False, inplace=True)
print("\nRough interaction scores (higher ≈ stronger interaction):")
display(pairs_df)


## 8. Permutation importance on held-out people

A second view of importance: shuffle one feature across the **test people** and see how much balanced accuracy falls. It measures what the model needs to score well on new people, not how it splits its output; correlated features hide each other here too, because the model can lean on the twin that was not shuffled.


In [ ]:
# T9.21 Permutation importance against mean |SHAP|
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test_s, y_test, scoring='balanced_accuracy', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
all_classes = np.abs(np.array(shap_exp.values)).mean(axis=(0, 2))        # mean |SHAP| over windows and activities
compare = pd.DataFrame({'permutation_drop': perm.importances_mean, 'mean_abs_shap': all_classes}, index=feature_cols)
compare['rank_permutation'] = compare['permutation_drop'].rank(ascending=False).astype(int)
compare['rank_shap'] = compare['mean_abs_shap'].rank(ascending=False).astype(int)
display(compare.sort_values('rank_shap').head(10).round(4))


In [ ]:
# T9.22 Report the explanations
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T9.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, tutorial="T9",
                 lesson={"base_value": base_value, "max_feature_correlation": max_corr})
else:
    print("Not connected: nothing sent. Run T9.00 with your code to use the coach.")


## 8. ✍️ Exercise — Global vs Local SHAP for one activity

Pick **one activity class** (e.g., "walking") and **one model** (the forest above, or a model of your own scored the same way first):

1. **Global view**
   - Identify the **top 10 features** by mean |SHAP|.

2. **Local view**
   - Find **3 correct** and **2 incorrect** test samples for that class (based on model predictions).
   - For each sample, generate a **waterfall plot** for that class.

3. **Compare & discuss (Optional, bonus)**
   - Do the **local important features** match the **global top features**?
   - For incorrect predictions, which features are pushing toward the predicted (wrong) class?

4. **Without a causal verb.** Rewrite your conclusion so that it says what the model used, not what makes a person walk.

If you are connected, ask the coach to review your answer to 3: it will ask whether the features that disagree are correlated.
